# CommitBench filtering

Loads the CommitBench dataset, filters it down to good training examples, and writes
`train.jsonl` / `validation.jsonl` / `test.jsonl`.

Filtering policy:
- drop diffs touching lockfiles / generated / minified files
- keep only messages matching Conventional Commits (`feat:`, `fix:`, `refactor:`, ...)
- drop low-content messages ("fix", "wip", "update", too short, ...)
- truncate oversized diffs per-file, prioritizing hunks over full context
- deduplicate within each split by (diff, message) hash
- cap over-represented commit types on **train only** (default: `fix` capped at 12 000)
- clean placeholder tokens (`<I>`, `<URL>`, `<EMAIL>`, `<HASH>`) from training messages

**Expect a low keep rate.** Raw CommitBench messages are real-world GitHub commits,
not Conventional-Commits-formatted — only ~5% naturally match a `feat:`/`fix:`/...
prefix. That's expected, not a bug: with ~1.17M train rows this still yields tens
of thousands of clean examples, plenty for LoRA fine-tuning.

**On Kaggle:** Settings -> Internet -> On (required to download the dataset and
packages). No GPU/accelerator needed for this notebook.

This notebook is self-contained (no dependency on the local `commit_model` package)
so it can run anywhere — locally, on Kaggle, or on Colab. Upload the resulting
`*.jsonl` files as input to `fine-tuning_Kaggle.ipynb`.

In [ ]:
# --- Dependencies ---
%pip install -q -U datasets tqdm

In [ ]:
import hashlib
import json
import os
import re
import shutil
from collections import Counter
from pathlib import Path

from datasets import load_dataset
from tqdm.auto import tqdm

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

# Kaggle only allows writes under /kaggle/working; detect it so this notebook
# also works unchanged when run locally or on Colab.
ON_KAGGLE = os.path.exists("/kaggle/working")
OUT_DIR = Path("/kaggle/working/data/processed") if ON_KAGGLE else Path("data/processed")

# Set an int for quick iteration, e.g. LIMIT_PER_SPLIT = 2000
LIMIT_PER_SPLIT = None

# Reproducibility seed (used for any shuffle operations)
RANDOM_SEED = 42

# Class rebalancing: cap over-represented types in the *train* split only.
# Keys are conventional-commit type prefixes; values are max examples kept.
# Only 'fix' is capped by default — it comprises >50% of valid examples and
# causes the model to default to 'fix' for ambiguous inputs.
# Set to {} to disable rebalancing entirely.
MAX_PER_TYPE: dict = {"fix": 12_000}

In [ ]:
# ---------------------------------------------------------------------------
# Filtering helpers (mirrors src/commit_model/diff_utils.py — kept inline so
# the notebook is self-contained and runs on Kaggle / Colab without the package)
# ---------------------------------------------------------------------------

CONVENTIONAL_PREFIX = re.compile(
    r"^(feat|fix|refactor|chore|docs|test|perf|style|build|ci)(\([\w./-]+\))?!?:\s+\S"
)

# Regex that matches the type prefix only (used for per-type counting)
TYPE_RE = re.compile(r"^(\w+)(\(.+?\))?!?:")

LOW_CONTENT_STOPLIST = {
    "fix", "wip", "update", "updates", "minor", "minor changes", "misc",
    "cleanup", "small fix", "typo", "changes", "stuff", "test", "tmp",
    "asdf", "oops", "fixup",
}

GENERATED_FILE_PATTERNS = [
    re.compile(p) for p in [
        r"package-lock\.json$", r"yarn\.lock$", r"pnpm-lock\.yaml$",
        r"Cargo\.lock$", r"Gemfile\.lock$", r"poetry\.lock$", r"uv\.lock$",
        r"\.min\.(js|css)$", r"\.map$", r"/dist/", r"/build/", r"/vendor/",
        r"\.svg$", r"\.snap$",
    ]
]

MAX_DIFF_CHARS = 4000
MAX_FILES_PER_DIFF = 6

# Placeholders injected by CommitBench into anonymised messages.
# Strip them from training data so the model never learns to generate them.
# (At inference time, postprocess.py also strips them — this is the training-side fix.)
_PLACEHOLDER_RE = re.compile(
    r"<(I|URL|EMAIL|HASH)>"
    r"|\(#<I>\)"
    r"|\b[0-9a-f]{7,40}\b"  # bare hex hashes that escaped the <HASH> anonymisation
    r"|https?://\S+"         # bare URLs that escaped the <URL> anonymisation
)


def is_generated_path(path: str) -> bool:
    return any(p.search(path) for p in GENERATED_FILE_PATTERNS)


def message_is_low_content(message: str) -> bool:
    first_line = message.strip().splitlines()[0] if message.strip() else ""
    has_prefix = CONVENTIONAL_PREFIX.match(first_line)
    subject = first_line.split(":", 1)[-1].strip() if has_prefix else first_line
    normalized = subject.lower().strip(" .!")
    return len(normalized) < 8 or normalized in LOW_CONTENT_STOPLIST


def split_diff_by_file(diff: str):
    """Split a unified diff into (path, hunk_text) chunks."""
    chunks = re.split(r"(?=^diff --git )", diff, flags=re.MULTILINE)
    files = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        match = re.search(r"^diff --git a/(\S+) b/(\S+)", chunk)
        path = match.group(2) if match else "unknown"
        files.append((path, chunk))
    return files


def truncate_diff(diff: str):
    """Drop generated/lockfile paths, cap file count, and cap total length."""
    files = split_diff_by_file(diff)
    kept = [(p, c) for p, c in files if not is_generated_path(p)]
    if not kept:
        return None
    kept = kept[:MAX_FILES_PER_DIFF]
    pieces = []
    for path, chunk in kept:
        share = max(MAX_DIFF_CHARS // max(len(kept), 1), 200)
        pieces.append(chunk[:share])
    return "".join(pieces)[:MAX_DIFF_CHARS]


def clean_training_message(message: str) -> str:
    """Strip CommitBench placeholder tokens from a training message.

    Does NOT apply the lowercase-first-word cosmetic from postprocess.py —
    that is inference-time only and should not alter training labels.
    Returns an empty string if nothing meaningful remains after cleaning.
    """
    cleaned = _PLACEHOLDER_RE.sub("", message)
    # Collapse multiple spaces and trim
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned).strip()
    # Collapse dangling punctuation left behind: 'fix: (#)' -> 'fix:'
    cleaned = re.sub(r":\s*[()#\[\]]+\s*$", "", cleaned).strip()
    return cleaned


def extract_type(message: str):
    """Return the commit type prefix string, or None."""
    m = TYPE_RE.match(message.strip())
    return m.group(1) if m else None

In [ ]:
# ---------------------------------------------------------------------------
# build_example — the central per-row filter function
# ---------------------------------------------------------------------------

def build_example(diff: str, message: str, seen_hashes: set, type_counts: dict, is_train: bool):
    """Return a filtered example dict, or None if the row should be dropped.

    Args:
        diff:        raw diff string from CommitBench
        message:     raw commit message from CommitBench
        seen_hashes: mutable set of (diff+message) hash strings — dedups within a split
        type_counts: mutable counter of kept examples per type — used for class rebalancing
        is_train:    True when processing the train split; class cap only applies then

    Returns:
        {"diff": str, "message": str} or None
    """
    message = message.strip()
    if not message:
        return None

    first_line = message.splitlines()[0]

    # --- Format check (must pass BEFORE cleaning to avoid accepting borderline messages) ---
    if not CONVENTIONAL_PREFIX.match(first_line):
        return None
    if message_is_low_content(message):
        return None

    # --- Diff quality check ---
    truncated = truncate_diff(diff)
    if truncated is None or len(truncated.strip()) < 20:
        return None

    # --- Placeholder cleaning (applied after format/quality gates) ---
    cleaned_message = clean_training_message(first_line)
    # If cleaning gutted the description (empty after the colon), reject the example
    if ":" not in cleaned_message:
        return None
    description_part = cleaned_message.split(":", 1)[1].strip()
    if len(description_part) < 4:
        return None
    # Re-validate format after cleaning
    if not CONVENTIONAL_PREFIX.match(cleaned_message):
        return None

    # --- Deduplication (within-split, by diff+message hash) ---
    h = hashlib.sha256((truncated + cleaned_message).encode()).hexdigest()[:16]
    if h in seen_hashes:
        return None
    seen_hashes.add(h)

    # --- Class rebalancing (train split only) ---
    commit_type = extract_type(cleaned_message)
    if is_train and commit_type and commit_type in MAX_PER_TYPE:
        if type_counts.get(commit_type, 0) >= MAX_PER_TYPE[commit_type]:
            return None

    # All checks passed — update counters and return the example
    if commit_type:
        type_counts[commit_type] = type_counts.get(commit_type, 0) + 1

    return {"diff": truncated, "message": cleaned_message}

In [ ]:
# ---------------------------------------------------------------------------
# Load CommitBench
# ---------------------------------------------------------------------------

print(f"Running on Kaggle: {ON_KAGGLE}")
print(f"Output dir:        {OUT_DIR.resolve()}")
print(f"LIMIT_PER_SPLIT:   {LIMIT_PER_SPLIT}")
print(f"MAX_PER_TYPE:      {MAX_PER_TYPE}")

try:
    ds = load_dataset("Maxscha/commitbench")
except Exception as e:
    raise RuntimeError(
        "Failed to download CommitBench. On Kaggle, make sure Settings -> Internet "
        "is turned ON, then re-run this cell."
    ) from e

In [ ]:
# ---------------------------------------------------------------------------
# Filter & write
# ---------------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name in ds.keys():
    split = ds[split_name]
    out_path = OUT_DIR / f"{split_name}.jsonl"
    is_train = split_name == "train"

    # Reset per-split state so deduplication does not cross splits
    seen_hashes: set = set()
    type_counts: dict = {}

    kept = 0
    seen = 0
    dedup_removed = 0
    placeholder_cleaned = 0

    with out_path.open("w") as f:
        for row in tqdm(split, desc=split_name, total=len(split)):
            seen += 1
            diff = row.get("diff") or ""
            message = (row.get("message") or "").strip()
            if not diff or not message:
                continue

            # Track whether this message had placeholders (for the counter)
            has_placeholder = bool(re.search(r"<(I|URL|EMAIL|HASH)>", message.splitlines()[0]))

            prev_hash_size = len(seen_hashes)
            example = build_example(diff, message, seen_hashes, type_counts, is_train)

            if example is None:
                # Check if it was rejected because it was a duplicate
                if len(seen_hashes) > prev_hash_size:
                    pass  # hash was added but rejected by a later gate — not a dup
                elif len(seen_hashes) == prev_hash_size and kept > 0:
                    # Hash was NOT added: either a dup or a pre-hash rejection.
                    # We can't easily distinguish here; dedup_removed is a lower bound.
                    pass
                continue

            if has_placeholder:
                placeholder_cleaned += 1

            f.write(json.dumps(example) + "\n")
            kept += 1
            if LIMIT_PER_SPLIT and kept >= LIMIT_PER_SPLIT:
                break

    print(f"\n{split_name}: kept {kept}/{seen} ({kept / max(seen, 1):.1%}) -> {out_path}")
    print(f"  Placeholder-cleaned messages included: {placeholder_cleaned}")
    if is_train and MAX_PER_TYPE:
        print(f"  Class rebalancing active (MAX_PER_TYPE={MAX_PER_TYPE})")
    print(f"  Per-type breakdown:")
    for t, n in sorted(type_counts.items(), key=lambda x: -x[1]):
        pct = n / max(kept, 1)
        capped = is_train and t in MAX_PER_TYPE and n >= MAX_PER_TYPE[t]
        cap_note = f" [CAPPED at {MAX_PER_TYPE[t]}]" if capped else ""
        print(f"    {t:<12} {n:>6}  ({pct:.1%}){cap_note}")

In [ ]:
# ---------------------------------------------------------------------------
# Sanity check — verify a few examples and assert output schema
# ---------------------------------------------------------------------------

sample_path = OUT_DIR / "train.jsonl"
print("Sample training examples:")
with sample_path.open() as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        row = json.loads(line)
        # Assert output schema: exactly two string keys
        assert set(row.keys()) == {"diff", "message"}, f"Unexpected keys: {row.keys()}"
        assert isinstance(row["diff"], str) and isinstance(row["message"], str)
        # Assert no placeholders survived into training data
        assert "<I>" not in row["message"], f"Placeholder <I> found in: {row['message']}"
        assert "<URL>" not in row["message"], f"Placeholder <URL> found in: {row['message']}"
        assert "<EMAIL>" not in row["message"], f"Placeholder <EMAIL> found in: {row['message']}"
        assert "<HASH>" not in row["message"], f"Placeholder <HASH> found in: {row['message']}"
        print(f"  [{i}] MESSAGE: {row['message']}")
        print(f"       DIFF:    {row['diff'][:200]}...\n")

print("Schema assertions passed.")

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle: zip output for download
# ---------------------------------------------------------------------------

if ON_KAGGLE:
    shutil.make_archive("/kaggle/working/commitbench-filtered", "zip", OUT_DIR)
    print("Wrote /kaggle/working/commitbench-filtered.zip")